# Complete Phase 4 development localization

Run the two code cells in order in the same GPU-enabled Kaggle kernel, with internet enabled for Git and the pinned CLIP checkpoint. Cell 1 verifies and, when necessary, installs `transformers==4.49.0` because Kaggle base images can change. All commands execute from `/kaggle/working/newpipeline/projects/logit_evidence_routing`.

Cell 1 fetches the latest `feat/iclr`, tests the new pipeline with synthetic inputs, and runs a NEW one-image paired-semantic smoke. This extends the already passed cached-map smoke by loading frozen CLIP on Kaggle, checking the paired projection/normalization against its global forward logits, and computing all 27 dense query maps. It does not load the 7B VLM or rerun Phase 2 extraction.

Cell 2 requires that matching smoke gate, scores all 240 development images, keeps train/validation summaries separate, verifies coverage and metrics, displays the validation tables and plots, and produces a ZIP containing reports and qualitative figures. It does not include model weights or dense score caches in the ZIP. The official CUB test split is never evaluated.

Outputs are commit-specific and resume using per-image score caches. If a cell is interrupted, rerun with the same checkout, protocol and input artifacts. Changed configuration or source hashes fail rather than silently mixing runs. A failed process stops the cell; do not bypass its gate. If the CUB input mount moved, specify only its small `parts/parts.txt` metadata path as described in Cell 1.

The fixed protocol is `configs/phase4_localization.json`. It contains the existing 26 attribute identities, text queries, explicit landmark-proxy mappings, K=16/32 and random seeds 0/1/2. These choices precede the Phase 4 full development result. Do not tune them using its validation outcomes.


In [ ]:
# Cell 1: fetch latest feat/iclr and run the NEW paired-semantic one-image smoke.
import os, sys, json, subprocess, importlib.metadata
from pathlib import Path
from IPython.display import display, Image, Markdown, FileLink

PROJECT = Path('/kaggle/working/newpipeline/projects/logit_evidence_routing')
os.chdir(PROJECT)
def git(*args):
    return subprocess.check_output(['git', *args], text=True).strip()
if git('status', '--porcelain', '--untracked-files=no'):
    raise RuntimeError('Preserve tracked repository edits before updating.')
subprocess.run(['git','fetch','--no-tags','origin',
                '+refs/heads/feat/iclr:refs/remotes/origin/feat/iclr'],check=True)
subprocess.run(['git','switch','feat/iclr'],check=True)
subprocess.run(['git','merge','--ff-only','origin/feat/iclr'],check=True)
HEAD = git('rev-parse','HEAD')
if HEAD != git('rev-parse','origin/feat/iclr'):
    raise RuntimeError('Checkout differs from latest fetched feat/iclr.')
print('Latest branch commit:',HEAD)

# Phase 4 intentionally pins the Transformers implementation used by the CLIP
# projection audit. Kaggle base images change over time, so make the requirement
# explicit instead of failing later inside a nested subprocess.
required_transformers = '4.49.0'
try:
    installed_transformers = importlib.metadata.version('transformers')
except importlib.metadata.PackageNotFoundError:
    installed_transformers = None
if installed_transformers != required_transformers:
    print(f'Installing transformers=={required_transformers}; found {installed_transformers!r}.', flush=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
         f'transformers=={required_transformers}'],
        check=True,
    )

# These paths match the completed Phase 1/2 runs. Override only if restored elsewhere.
COMMON = ['--stage-cache','/kaggle/working/phase2_stage_cache',
          '--localizer-cache','/kaggle/working/phase1b_corrected/cache',
          '--phase1-gate','/kaggle/working/phase1b_corrected/results/phase1_gate.json']
# Resolve the small CUB part vocabulary recorded by the completed Phase 2 run.
stage_cache = Path(COMMON[1])
localizer_cache = Path(COMMON[3])
phase1_gate = Path(COMMON[5])
stage_config = stage_cache/'run_config.json'
required_inputs = [stage_config, stage_cache/'index.json', stage_cache/'validation_report.json',
                   localizer_cache/'extraction_config.json', localizer_cache/'records', phase1_gate]
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError('Missing completed Phase 1/2 input(s):\n' + '\n'.join(missing_inputs))
stage_metadata = json.loads(stage_config.read_text())
PARTS = Path(stage_metadata['dataset']['root'])/'parts/parts.txt'
if not PARTS.is_file():
    raise FileNotFoundError(
        f'Phase 2 recorded the CUB part vocabulary at {PARTS}, but it is not mounted. '
        "Add the original CUB dataset as a Kaggle input or replace PARTS with that input's parts/parts.txt."
    )
COMMON += ['--part-vocabulary', str(PARTS)]
OUT = Path('/kaggle/working') / f'phase4_localization_{HEAD[:12]}'
SMOKE, FULL = OUT/'smoke', OUT/'development'
ENV = dict(os.environ, PYTHONPATH='src', PYTHONUNBUFFERED='1',
           HF_HUB_OFFLINE='0', TRANSFORMERS_OFFLINE='0',
           CUBLAS_WORKSPACE_CONFIG=':4096:8', MPLCONFIGDIR='/kaggle/working/.phase4_matplotlib')
def run(*args):
    command = [sys.executable, *map(str,args)]
    print('\n$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command, env=ENV, cwd=PROJECT, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
    if returncode:
        raise RuntimeError(
            f'Child command failed with exit code {returncode}. '
            'The complete child traceback is printed immediately above.'
        )
run('-c', "import torch, transformers, matplotlib; "
          "print('Runtime:', 'torch='+torch.__version__, 'transformers='+transformers.__version__, "
          "'matplotlib='+matplotlib.__version__, 'CUDA='+str(torch.cuda.is_available()))")
if not __import__('torch').cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Kaggle before running Phase 4.')
run('scripts/run_phase4_localization.py','--help')
run('scripts/validate_phase4_localization.py','--help')
# This test suite uses synthetic records and fake model outputs; no pretrained downloads.
run('-m','unittest','discover','-s','tests','-p','test_phase4.py','-v')
run('scripts/run_phase4_localization.py','--mode','smoke',*COMMON,'--output-dir',SMOKE)
run('scripts/validate_phase4_localization.py','--output-dir',SMOKE,
    '--bundle',OUT/'phase4_semantic_smoke_bundle.zip')
display(Markdown((SMOKE/'PHASE4_RESULTS.md').read_text()))
display(Image(filename=str(SMOKE/'localization_overview.png')))
for panel in sorted((SMOKE/'qualitative').glob('*.png')):
    display(Image(filename=str(panel)))
print('Semantic smoke complete. Cell 2 checks this gate before running all 240 development images.')


In [ ]:
# Cell 2: full development localization, independent validation, plots and result ZIP.
os.chdir('/kaggle/working/newpipeline/projects/logit_evidence_routing')
run('scripts/run_phase4_localization.py','--mode','development',*COMMON,
    '--smoke-dir',SMOKE,'--output-dir',FULL)
BUNDLE = OUT/'phase4_development_results_bundle.zip'
run('scripts/validate_phase4_localization.py','--output-dir',FULL,'--bundle',BUNDLE)
report=json.loads((FULL/'phase4_run_report.json').read_text())
if report['images']!=240 or report['split_counts']!={'train':160,'val':80}:
    raise RuntimeError('Development coverage differs from the fixed 160/80 split.')
display(Markdown((FULL/'PHASE4_RESULTS.md').read_text()))
display(Image(filename=str(FULL/'localization_overview.png')))
display(Image(filename=str(FULL/'selector_agreement.png')))
print('Phase 4 results:',FULL)
print('Object metric rows:',report['object_metric_rows'],'(expected 3,840)')
print('Attribute eligibility rows:',report['eligibility_rows'],'(expected 6,240)')
print('Eligible image/attribute pairs:',report['eligible_image_attribute_pairs'])
print('Attribute metric rows:',report['attribute_metric_rows'],'(18 per eligible pair)')
print('Official test images used:',report['official_test_images_used'])
print('Download this reports/plots ZIP and attach it for scientific review. It excludes model/score tensors:')
display(FileLink(str(BUNDLE)))


# What the results mean

There are two matched evaluations. Object localization compares Vision-CLS attention, LLM attention, corrected generic bird/birds Logit Lens, fusion, dense CLIP bird-query similarity, and random selection. Attribute localization adds the dense query for each named attribute, evaluated only on sufficiently certain positive labels with relevant visible in-crop landmark proxies. Generic attention/bird maps remain labeled generic controls; they are not attribute-conditioned maps.

The dense model is `openai/clip-vit-large-patch14-336` at revision `ce19dc912ca5cd21c8a653c79e251e808ccabcd1`. It computes cosine similarity between text embeddings and final patch states after CLIP's paired post-layer-normalization and visual projection. CLIP was trained with global image/text alignment; applying its head to contextual patches is a diagnostic, not dense-supervised segmentation. The RGB inputs are the already cropped uint8 images stored by Phase 1; no second resize/crop occurs.

Metrics include broad-box concentration, patch recall/IoU, top-1 pointing, visible/relevant-part patch recall, top-1 distance, selector agreement and paired differences versus random and the generic dense object query. Distinct random seeds are averaged within image. Macro attribute scores weight evaluable attributes equally, and the support table retains all 26 selected attributes, including those with zero eligible images. Missing/occluded parts are unavailable rather than zeros. SD describes image variability; no final-test confidence intervals are claimed.

The qualitative figures use fixed IDs: the smoke training image and the six smallest validation IDs. Each attribute panel uses the first eligible attribute in policy order, never the highest-scoring outcome. Heatmaps are independently scaled for display; cyan marks Top-32, yellow the top-1 patch, and green the ground-truth landmark proxies and bird box.

After both cells pass, attach `phase4_development_results_bundle.zip`. The numeric and qualitative results must be reviewed before Phase 5. Do not infer a causal bottleneck, evidence relocation, or VLM utilization from these localization measurements alone.

Implementation validation performed locally uses synthetic tensors/metadata and mocked pretrained scoring, plus rendered synthetic figures. No pretrained model weights, CUB data, actual VLM extraction, or actual probe experiments were downloaded/run locally. Real CLIP execution and Phase 4 measurements remain Kaggle work.
